# OLAP · 00 · Setup shared by every OLAP notebook

Every OLAP notebook starts with `%run ./00_olap_setup`. This notebook:

1. runs `../00_common` (catalog, table helpers) — the OLAP folder sits inside `notebooks/`, so the ETL helpers are one level up;
2. refuses to run on gold data whose latest validation failed;
3. builds two **temporary views** over the star schema, so every query reads like a cube;
4. provides `olap(name, sql)`, which runs a query, shows it, and saves the answer as a table in the `olap` schema.

Saving each answer as `workspace.olap.<name>` means the findings notebook, a Databricks dashboard and the report all read **the same numbers** without re-running the queries.

| Schema | Holds |
|---|---|
| `gold` | The star schema (input to OLAP, never changed here) |
| `olap` | One table per OLAP answer, overwritten on every OLAP run |

In [0]:
%run ../00_common

In [0]:
OLAP_SCHEMA = "olap"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{OLAP_SCHEMA}")

def otname(name):
    return f"{CATALOG}.{OLAP_SCHEMA}.{name}"

def olap(name, sql, show=True):
    """Run one OLAP query, save the answer as olap.<name>, display it, and return it as pandas."""
    df = spark.sql(sql)
    df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(otname(name))
    out = spark.table(otname(name))
    if show:
        display(out)
    return out.toPandas()

def oread(name):
    return spark.table(otname(name)).toPandas()

### Guard: only analyse a validated load
If the latest run of `40_gold_validate` has any FAIL, the numbers below could be wrong. Stop and fix the pipeline first.

In [0]:
if table_exists("audit", "validation_results"):
    last = spark.sql(f"""
        SELECT run_id, count_if(result = 'FAIL') AS fails, count(*) AS checks
        FROM {tname('audit', 'validation_results')}
        WHERE run_id = (SELECT max_by(run_id, checked_at) FROM {tname('audit', 'validation_results')})
        GROUP BY run_id""").first()
    print(f"Latest validated run: {last['run_id']} · {last['checks']} checks · {last['fails']} failed")
    assert last["fails"] == 0, "The latest validation run has failures. Fix the pipeline before running OLAP."
else:
    print("WARNING: audit.validation_results not found. Run 40_gold_validate first.")

### The cube views
`v_event_loss` joins the transaction fact to all four dimensions at the fact's grain (**one row per in-scope ICA catastrophe event, primary state**). Nothing is aggregated here, so every query below can roll up, drill down, slice, dice or pivot along any dimension attribute.

**Dimension hierarchies used for roll-up and drill-down**

| Dimension | Hierarchy (coarse → fine) |
|---|---|
| Date | calendar year → quarter → month (also: Australian financial year; season) |
| Hazard | hazard family → hazard group |
| Location | country → state type → state |

**Measures** (all on `fact_event_loss`)

| Measure | Additive? | Note |
|---|---|---|
| `loss_m` — normalised insured loss, 2022 AUD millions | Yes | ICA's normalisation. Events from 2023 on are in their own year's dollars. |
| `nominal_loss_m` — original insured loss, AUD millions | Yes | As reported at the time |
| `total_claims` and the claim lines | Yes | Recorded for only some events: always report how many events have a value |
| `events` = `count(*)` | Yes | |
| Loss per event, loss per claim, loss per resident | **No** | Always computed as a ratio of sums, never averaged |

`v_state_year` is the periodic snapshot (state × year) with population and median income. Population and income are **non-additive across time**, so queries average them, never sum them.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_event_loss AS
SELECT e.event_key, e.event_id, e.event_name, e.declaration_status, e.state_source, e.multi_state_flag, e.de_matched,
       d.full_date AS start_date, d.calendar_year, d.calendar_quarter, d.month_number, d.month_name,
       d.au_financial_year, d.season,
       h.hazard_family, h.hazard_group,
       l.country, l.state_type, l.state_code, l.state_name,
       f.normalised_loss_2022_aud / 1e6 AS loss_m,
       f.original_loss_aud / 1e6        AS nominal_loss_m,
       f.total_claims, f.domestic_building_claims, f.domestic_content_claims, f.domestic_motor_claims, f.domestic_other_claims,
       coalesce(f.commercial_property_claims, 0) + coalesce(f.commercial_motor_claims, 0) + coalesce(f.commercial_bi_claims, 0)
         + coalesce(f.commercial_other_claims, 0) + coalesce(f.commercial_crop_claims, 0) AS commercial_claims,
       f.duration_days, f.deaths, f.injuries, f.homes_destroyed
FROM {tname('gold', 'fact_event_loss')} f
JOIN {tname('gold', 'dim_event')}    e ON e.event_key    = f.event_key
JOIN {tname('gold', 'dim_date')}     d ON d.date_key     = f.start_date_key
JOIN {tname('gold', 'dim_hazard')}   h ON h.hazard_key   = f.hazard_key
JOIN {tname('gold', 'dim_location')} l ON l.location_key = f.location_key
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_state_year AS
SELECT l.state_code, l.state_name, l.state_type, d.calendar_year AS year,
       s.population, s.population_imputed, s.median_income_aud, s.income_imputed
FROM {tname('gold', 'fact_state_year')} s
JOIN {tname('gold', 'dim_location')} l ON l.location_key = s.location_key
JOIN {tname('gold', 'dim_date')}     d ON d.date_key     = s.date_key
""")

chk = spark.sql("SELECT count(*) AS events, round(sum(loss_m), 1) AS loss_m FROM v_event_loss").first()
fact_n = spark.table(tname("gold", "fact_event_loss")).count()
print(f"v_event_loss: {chk['events']} events (fact has {fact_n}) · total normalised loss ${chk['loss_m']:,.1f}m")
assert chk["events"] == fact_n, "The cube view lost rows: a dimension join is missing a key."

### Chart style (used by the findings notebook)
One categorical colour per hazard, in a fixed order, so a hazard keeps its colour in every chart. Single-measure charts use one blue.

In [0]:
HAZARD_ORDER = ["Flood", "Hail", "Storm", "Cyclone", "Bushfire", "Earthquake"]
HAZARD_COLOURS = dict(zip(HAZARD_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
BLUE, BLUE_LIGHT, GREY = "#2a78d6", "#9ec5f4", "#c3c2b7"
INK, INK_2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"